# Entrenamiento y evaluación

Un run por ejecución: se elige `RUN_ID` en la sección 3 y se corre todo el notebook. Funciona igual en **Kaggle**, **Colab** o **local**: detecta la plataforma, ubica el dataset y genera ahí los `data.yaml` con las rutas de esa máquina.

| Run | Modelo | imgsz | Train | Responde |
|---|---|---|---|---|
| R1 | `yolo11n` | 640 | con negativos | Pareto capacidad/latencia (baseline) |
| R2 | `yolo11s` | 640 | con negativos | Pareto capacidad/latencia |
| R3 | `yolo11m` | 640 | con negativos | Pareto (**opcional**) |
| R4 | mejor tamaño | 960 | con negativos | **H1**: resolución |
| R5 | mejor tamaño | 640 | **sin** negativos | **H2**: negativos |

Nombre de cada run: `{modelo}_{imgsz}_{neg|noneg}_s{seed}`, por ejemplo `yolo11n_640_neg_s42`.

**Requisitos por plataforma**

- **Kaggle:** *Add Input* → dataset `sayedgamal99/smoke-fire-detection-yolo`; *Settings* → Accelerator GPU y **Internet on** (para instalar ultralytics y bajar los pesos preentrenados).
- **Colab:** Runtime con GPU. El dataset se baja con `kagglehub`; si no hay credenciales, cargar `KAGGLE_USERNAME` y `KAGGLE_KEY` en los *Secrets* de Colab. Los runs se guardan en Drive para sobrevivir a desconexiones. Si Drive no se puede montar (por ejemplo, Colab usado desde VS Code), se guardan en `/content/incendios-tp`: funciona igual, pero si la sesión se corta se pierde el run, y hay que **bajar el zip del final antes de cerrar**.
- **Local:** haber corrido `descarga.ipynb`. Sin GPU solo sirve `SMOKE_TEST = True`.

Si la sesión se corta, volver a correr el notebook con el mismo `RUN_ID`: retoma desde `last.pt`.

## 1. Plataforma y dependencias

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Kaggle se reconoce por su variable de entorno, no por /kaggle/input: Colab también puede tener esa carpeta
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
# Colab: el módulo ya cargado, o las variables de entorno que define el runtime
IN_COLAB = "google.colab" in sys.modules or any(k.startswith("COLAB_") for k in os.environ)
PLATFORM = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"

ULTRALYTICS_VERSION = "8.4.157"  # misma versión que el entorno local (environment.yml)
if PLATFORM != "local":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}", "kagglehub"],
        check=True,
    )

DRIVE_OK = False
if PLATFORM == "colab":
    # Drive permite retomar un run en otra sesión. Si no se puede montar (Colab desde VS Code,
    # ventana de autorización bloqueada, varias cuentas de Google abiertas), se sigue en el disco
    # de la sesión: funciona igual, pero hay que bajar el zip del final antes de cerrar.
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OK = True
    except Exception as e:
        print(f"[!] No se pudo montar Drive ({e}). Se guarda en /content/incendios-tp: "
              "bajar el zip del final antes de cerrar la sesión.")

WORK_DIR = {
    "kaggle": Path("/kaggle/working"),
    "colab": Path("/content/drive/MyDrive/incendios-tp" if DRIVE_OK else "/content/incendios-tp"),
    "local": Path.cwd(),
}[PLATFORM]
RUNS_DIR = WORK_DIR / "runs"
RESULTS_DIR = WORK_DIR / "results"
RUNTIME_CFG_DIR = RUNS_DIR / "_data"  # yaml con rutas absolutas de esta máquina: no se versiona
for d in (RUNS_DIR, RESULTS_DIR, RUNTIME_CFG_DIR):
    d.mkdir(parents=True, exist_ok=True)

import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE == 0 else "cpu"
print(f"Plataforma: {PLATFORM} | ultralytics {ultralytics.__version__} | torch {torch.__version__} | device: {DEVICE_NAME}")
print("Resultados en:", WORK_DIR)

## 2. Dataset y configs

Ubicamos la carpeta que contiene `train/`, `val/` y `test/` (cada una con `images/` y `labels/`), repetimos los chequeos de `descarga.ipynb` (conteos del split oficial y orden de clases) y generamos dos yaml en `runs/_data/`:

- `data_neg.yaml`: split oficial completo.
- `data_noneg.yaml`: **train** sin imágenes "none" (R5); val y test idénticos.

Warnings esperados de Ultralytics:

- Descarta 4 imágenes de test con coordenadas > 1,01 (ver `descarga.ipynb`, sección 7).
- `corrupt JPEG restored and saved`: 32 JPEG truncados de D-Fire se reparan en el lugar la primera vez. Por eso en Kaggle el dataset se copia a `/tmp`: en `/kaggle/input` (solo lectura) esas imágenes se descartarían.

In [ ]:
DATASET_SLUG = "sayedgamal99/smoke-fire-detection-yolo"
SPLITS = ("train", "val", "test")
OFFICIAL_SPLIT = {"train": 14122, "val": 3099, "test": 4306}
CLASS_NAMES = {0: "smoke", 1: "fire"}
OFFICIAL_TRAIN_BOXES = {0: 7794, 1: 9638}  # verificado en descarga.ipynb

def find_split_root(base: Path) -> Path | None:
    """Carpeta que contiene train/, val/ y test/, cada una con images/ y labels/."""
    for train_dir in base.rglob("train"):
        root = train_dir.parent
        if all((root / s / sub).is_dir() for s in SPLITS for sub in ("images", "labels")):
            return root
    return None

def descargar_kagglehub() -> Path | None:
    import kagglehub
    return find_split_root(Path(kagglehub.dataset_download(DATASET_SLUG)))

if PLATFORM == "kaggle":
    # /kaggle/input es de solo lectura: Ultralytics no puede reparar los JPEG truncados y descarta esas
    # imágenes (11 de test, 21 de train), así val/test no serían los mismos que en Colab o local.
    # Copiamos a /tmp (~3 GB, no cuenta para el Output de Kaggle).
    import shutil
    src = find_split_root(Path("/kaggle/input")) or descargar_kagglehub()
    SPLIT_ROOT = Path("/tmp/dfire")
    if src is not None and not SPLIT_ROOT.exists():
        shutil.copytree(src, SPLIT_ROOT)
    SPLIT_ROOT = SPLIT_ROOT if src is not None else None
elif PLATFORM == "colab":
    SPLIT_ROOT = descargar_kagglehub()
else:
    SPLIT_ROOT = find_split_root(Path.cwd() / "data")
assert SPLIT_ROOT is not None, (
    f"No se encontró D-Fire (plataforma detectada: {PLATFORM}, carpeta actual: {Path.cwd()}). "
    "Kaggle: Add Input → sayedgamal99/smoke-fire-detection-yolo. Local: correr descarga.ipynb. "
    "Si estás en Colab y dice 'local', avisar: falló la detección de Colab."
)
print("Dataset:", SPLIT_ROOT)

# Chequeos de sanidad
for s, n in OFFICIAL_SPLIT.items():
    n_img = sum(1 for _ in (SPLIT_ROOT / s / "images").iterdir())
    assert n_img == n, f"{s}: {n_img} imágenes, se esperaban {n}"

train_label_text = {p.stem: p.read_text() for p in (SPLIT_ROOT / "train" / "labels").glob("*.txt")}
box_counts = {0: 0, 1: 0}
for text in train_label_text.values():
    for line in text.splitlines():
        if line.strip():
            box_counts[int(line.split()[0])] += 1
assert box_counts == OFFICIAL_TRAIN_BOXES, f"Cajas por índice en train {box_counts} != {OFFICIAL_TRAIN_BOXES}"

# Configs con rutas absolutas de esta máquina
train_pos = sorted(
    p for p in (SPLIT_ROOT / "train" / "images").iterdir() if train_label_text.get(p.stem, "").strip()
)
noneg_list = RUNTIME_CFG_DIR / "train_noneg.txt"
noneg_list.write_text("\n".join(p.as_posix() for p in train_pos) + "\n", encoding="utf-8")

names_block = "\n".join(f"  {i}: {n}" for i, n in CLASS_NAMES.items())
val_test = (
    f"val: {(SPLIT_ROOT / 'val' / 'images').as_posix()}\n"
    f"test: {(SPLIT_ROOT / 'test' / 'images').as_posix()}\n"
    f"\nnames:\n{names_block}\n"
)
DATA_YAMLS = {
    "neg": RUNTIME_CFG_DIR / "data_neg.yaml",
    "noneg": RUNTIME_CFG_DIR / "data_noneg.yaml",
}
DATA_YAMLS["neg"].write_text(f"train: {(SPLIT_ROOT / 'train' / 'images').as_posix()}\n" + val_test, encoding="utf-8")
DATA_YAMLS["noneg"].write_text(f"train: {noneg_list.as_posix()}\n" + val_test, encoding="utf-8")

print(f"Train con negativos: {OFFICIAL_SPLIT['train']} imágenes | sin negativos: {len(train_pos)}")

## 3. Configuración del run

Lo único que se toca entre runs es `RUN_ID` (y `BEST_SIZE` cuando R1–R3 estén cargados en `master.csv`).

- `EPOCHS = 50` con `patience = EPOCHS`: sin early stopping, todos los runs entrenan lo mismo y son comparables.
- `BATCH = 16`: bajar a 8 si R3 o R4 (960 px) dan out-of-memory, y anotarlo.
- Aumentos de datos: los valores por defecto de Ultralytics, explícitos para poder citarlos en el paper (mosaic, HSV, volteo horizontal, escala). Solo se aplican en train.

In [ ]:
SEED = 42
RUN_ID = "R1"        
BEST_SIZE = "n"      
EPOCHS = 50
BATCH = 16
SMOKE_TEST = False   

RUNS = {
    "R1": {"size": "n", "imgsz": 640, "neg": True},
    "R2": {"size": "s", "imgsz": 640, "neg": True},
    "R3": {"size": "m", "imgsz": 640, "neg": True},
    "R4": {"size": BEST_SIZE, "imgsz": 960, "neg": True},
    "R5": {"size": BEST_SIZE, "imgsz": 640, "neg": False},
}

AUGMENT = dict(
    mosaic=1.0, close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, scale=0.5, translate=0.1,
)

cfg = RUNS[RUN_ID]
MODEL = f"yolo11{cfg['size']}"
IMGSZ = cfg["imgsz"]
TRAIN_SET = "neg" if cfg["neg"] else "noneg"
RUN_NAME = f"{MODEL}_{IMGSZ}_{TRAIN_SET}_s{SEED}" + ("_smoke" if SMOKE_TEST else "")
RUN_DIR = RUNS_DIR / RUN_NAME
EPOCHS_RUN = 1 if SMOKE_TEST else EPOCHS
if SMOKE_TEST:
    BATCH = 4  # en CPU, torch 2.14 se cae (segfault) con batch 16 a 640 px

if DEVICE == "cpu" and not SMOKE_TEST:
    raise RuntimeError("Sin GPU: un run completo en CPU lleva días. Usar Kaggle/Colab o SMOKE_TEST = True.")
print(f"{RUN_ID} → {RUN_NAME} | data: {DATA_YAMLS[TRAIN_SET].name} | epochs: {EPOCHS_RUN} | batch: {BATCH}")

## 4. Entrenamiento

Tres casos según lo que haya en `runs/<RUN_NAME>/`:

- `results.csv` con todas las épocas → el run ya terminó, se saltea.
- `weights/last.pt` sin terminar → se retoma (`resume=True`).
- Nada → entrenamiento desde los pesos COCO de `yolo11{n,s,m}.pt`.

In [ ]:
def epochs_done(run_dir: Path) -> int:
    csv = run_dir / "results.csv"
    return len(pd.read_csv(csv)) if csv.exists() else 0

last_pt = RUN_DIR / "weights" / "last.pt"
best_pt = RUN_DIR / "weights" / "best.pt"

if epochs_done(RUN_DIR) >= EPOCHS_RUN and best_pt.exists():
    print(f"{RUN_NAME} ya está entrenado ({epochs_done(RUN_DIR)} épocas), se saltea.")
elif last_pt.exists():
    print(f"Retomando {RUN_NAME} desde la época {epochs_done(RUN_DIR)}")
    YOLO(last_pt).train(resume=True)
else:
    YOLO(f"{MODEL}.pt").train(
        data=str(DATA_YAMLS[TRAIN_SET]),
        epochs=EPOCHS_RUN,
        patience=EPOCHS_RUN,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        deterministic=True,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        fraction=0.02 if SMOKE_TEST else 1.0,
        plots=True,
        **AUGMENT,
    )

## 5. Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

hist = pd.read_csv(RUN_DIR / "results.csv")
hist.columns = hist.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for col in ("train/box_loss", "train/cls_loss", "val/box_loss", "val/cls_loss"):
    axes[0].plot(hist["epoch"], hist[col], label=col, ls="-" if col.startswith("train") else "--")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].set_title(f"Pérdidas — {RUN_NAME}")
axes[0].legend()

for col, label in (("metrics/mAP50(B)", "mAP50"), ("metrics/mAP50-95(B)", "mAP50-95")):
    axes[1].plot(hist["epoch"], hist[col], label=label)
axes[1].set_xlabel("Época")
axes[1].set_ylabel("mAP (val)")
axes[1].set_title(f"mAP en validación — {RUN_NAME}")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RUN_DIR / "curvas.png", dpi=150, bbox_inches="tight")

## 6. Evaluación y `results/master.csv`

`evaluate` toma un `best.pt` y un split y devuelve siempre las mismas métricas: mAP50 y mAP50-95 global y por clase, P y R por clase, y ms/imagen de inferencia. Se evalúa siempre contra `data_neg.yaml`: val y test son los mismos para todos los runs, incluido R5.

Cada run es **una fila** de `master.csv`, con las métricas de val y test como columnas (`val_*`, `test_*`). Si el run ya estaba, la fila se reemplaza.

Las ms/imagen de esta celda son orientativas (batch de validación, GPU de la plataforma): la latencia del paper sale de `benchmark.py` (batch 1, GPU y CPU).

In [ ]:
from datetime import date

def evaluate(weights: Path, split: str, imgsz: int) -> dict:
    m = YOLO(weights).val(
        data=str(DATA_YAMLS["neg"]),
        split=split,
        imgsz=imgsz,
        batch=BATCH,
        device=DEVICE,
        project=str(RUN_DIR),
        name=f"eval_{split}",
        exist_ok=True,
        plots=True,
        verbose=False,
    )
    row = {"mAP50": m.box.map50, "mAP50-95": m.box.map, "P": m.box.mp, "R": m.box.mr}
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        name = m.names[int(c)]
        row |= {f"mAP50_{name}": ap50, f"mAP50-95_{name}": ap, f"P_{name}": p, f"R_{name}": r}
    row["ms_inferencia"] = m.speed["inference"]
    return {f"{split}_{k}": round(float(v), 4) for k, v in row.items()}

EVAL_SPLITS = ("val",) if SMOKE_TEST else ("val", "test")
row = {
    "run": RUN_NAME, "run_id": RUN_ID, "modelo": MODEL, "imgsz": IMGSZ, "train_set": TRAIN_SET,
    "seed": SEED, "epochs": epochs_done(RUN_DIR), "batch": BATCH,
    "plataforma": PLATFORM, "device": DEVICE_NAME, "ultralytics": ultralytics.__version__,
    "fecha": date.today().isoformat(),
}
for split in EVAL_SPLITS:
    row |= evaluate(best_pt, split, IMGSZ)

pd.Series(row).to_frame(RUN_NAME)

In [ ]:
# Referencia del plan: YOLOv8n sobre D-Fire da mAP50 ≈ 0,75 y mAP50-95 ≈ 0,43.
# Si el baseline queda cerca de 0,4 de mAP50, hay un bug en el data.yaml o en el split.
if not SMOKE_TEST and row["val_mAP50"] < 0.6:
    print(f"[!] val mAP50 = {row['val_mAP50']:.3f}: muy por debajo de la referencia (~0,75). Revisar antes de seguir.")
else:
    print(f"val mAP50 = {row['val_mAP50']:.3f} | val mAP50-95 = {row['val_mAP50-95']:.3f}")

In [ ]:
MASTER = RESULTS_DIR / ("master_smoke.csv" if SMOKE_TEST else "master.csv")

master = pd.read_csv(MASTER) if MASTER.exists() else pd.DataFrame()
if not master.empty:
    master = master[master["run"] != RUN_NAME]
master = pd.concat([master, pd.DataFrame([row])], ignore_index=True).sort_values("run_id")
master.to_csv(MASTER, index=False)
print("Actualizado:", MASTER)
master

## 7. Gráficos de Ultralytics sobre test

Matriz de confusión normalizada y curva precision-recall por clase, generadas por `val()` en `runs/<RUN_NAME>/eval_test/`.

In [ ]:
from IPython.display import Image as IPImage, display

eval_dir = RUN_DIR / f"eval_{EVAL_SPLITS[-1]}"
for pattern in ("confusion_matrix_normalized.png", "*PR_curve.png"):
    for p in sorted(eval_dir.glob(pattern)):
        print(p.name)
        display(IPImage(filename=str(p), width=700))

## 8. Llevar el resultado al repo

- **Kaggle:** lo que queda en `/kaggle/working` aparece en la pestaña *Output* del notebook al guardar la versión (*Save Version → Quick Save* no corre todo; usar *Save & Run All* si se quiere el run completo en background).
- **Colab:** queda en `Drive/incendios-tp/`, o en `/content/incendios-tp/` si Drive no se montó (bajarlo desde el panel *Files*, o en VS Code desde el explorador de archivos del kernel, antes de cerrar la sesión).
- **Local:** queda directamente en `runs/` y `results/`.

La celda de abajo arma `<RUN_NAME>_resultados.zip`. Para incorporarlo al repo:

1. Descomprimirlo en la raíz del repo: crea `runs/<RUN_NAME>/` (con `weights/best.pt`) y `results/master_<RUN_NAME>.csv`.
2. Correr la última celda (`merge_master`) en local, el mismo día (regla 1 del plan): agrega o reemplaza la fila de ese run en `results/master.csv` sin tocar las de los demás.

`best.pt` queda en disco para evaluación, benchmark y demo, pero **no** se versiona (`runs/**/weights/` está en `.gitignore`). `args.yaml` del run guarda las rutas de Colab/Kaggle: es solo registro, no hace falta tocarlo.

In [ ]:
import zipfile

# Zip del run para bajarlo de Kaggle/Colab de una sola vez. Se descomprime en la raíz del repo:
#   runs/<RUN_NAME>/...                (con weights/best.pt; last.pt solo sirve para retomar acá)
#   results/master_<RUN_NAME>.csv      (solo la fila de este run, para combinar con merge_master)
archive = WORK_DIR / f"{RUN_NAME}_resultados.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.rglob("*"):
        if p.is_file() and p.name != "last.pt":
            z.write(p, Path("runs") / p.relative_to(RUNS_DIR))
    z.writestr(f"results/master_{RUN_NAME}.csv", pd.DataFrame([row]).to_csv(index=False))
print("Zip:", archive, f"({archive.stat().st_size / 1e6:.1f} MB)")

### Combinar resultados descargados (en local)

Independiente del resto del notebook: se puede correr sola. Toma todos los `results/master_*.csv` descomprimidos y los combina en `results/master.csv` (una fila por run; si el run ya estaba, gana la versión nueva). Los parciales se borran después de combinarlos.

In [1]:
from pathlib import Path
import pandas as pd

def merge_master(results_dir: Path = Path.cwd() / "results") -> pd.DataFrame:
    master_path = results_dir / "master.csv"
    parts = sorted(results_dir.glob("master_*.csv"))
    frames = [pd.read_csv(master_path)] if master_path.exists() else []
    frames += [pd.read_csv(p) for p in parts]
    if not frames:
        print("No hay resultados para combinar en", results_dir)
        return pd.DataFrame()
    master = (
        pd.concat(frames, ignore_index=True)
        .drop_duplicates(subset="run", keep="last")
        .sort_values("run_id")
        .reset_index(drop=True)
    )
    master.to_csv(master_path, index=False)
    for p in parts:
        p.unlink()
    print(f"{master_path}: {len(master)} runs ({len(parts)} combinados)")
    return master

merge_master()

c:\Proyectos\CV2\detección_incendios\results\master.csv: 5 runs (1 combinados)


,run,run_id,modelo,imgsz,train_set,seed,epochs,batch,plataforma,device,...,test_P_fire_sin_clf,test_R_fire_sin_clf,test_FP_por_img_neg_sin_clf,test_pct_img_neg_con_FP_sin_clf,test_P_smoke_con_clf,test_R_smoke_con_clf,test_P_fire_con_clf,test_R_fire_con_clf,test_FP_por_img_neg_con_clf,test_pct_img_neg_con_FP_con_clf
0,clasificador_yolo11n_640_noneg_s42,CLF,yolo11n-cls,224,recortes (yolo11n_640_noneg_s42),42,30,64,colab,NVIDIA L4,...,0.7796,0.5532,0.007,0.6484,0.8504,0.6726,0.7858,0.5441,0.001,0.0998
1,yolo11n_640_neg_s42,R1,yolo11n,640,neg,42,50,16,colab,NVIDIA L4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,yolo11m_640_neg_s42,R3,yolo11m,640,neg,42,50,16,colab,NVIDIA L4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,yolo11n_960_neg_s42,R4,yolo11n,960,neg,42,50,16,colab,NVIDIA L4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,yolo11n_640_noneg_s42,R5,yolo11n,640,noneg,42,50,16,colab,NVIDIA L4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
